In [28]:
# Загрузка и парсинг архива

import asyncio
import csv
import logging
import re
from dataclasses import dataclass
from datetime import date, datetime, timedelta, UTC
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import urljoin, urlparse

import aiohttp
from bs4 import BeautifulSoup, Tag

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S",
)
logger = logging.getLogger("LentaArchiveParser")


# =========================
# НАСТРОЙКИ
# =========================

BASE_URL = "https://lenta.ru"

# Даты
START_DATE = "01.01.2024"
END_DATE = "31.01.2024"

# Папка и имя файла
OUTPUT_DIR = "data/raw"
OUTPUT_FILENAME = "lenta_news_jan_2024.csv"

# Основные рубрики Ленты

RUBRIC_TO_SLUG = {
    "Россия": "russia",
    "Мир": "world",
    "Бывший СССР": "ussr",
    "Экономика": "economics",
    "Силовые структуры": "forces",
    "Наука и техника": "science",
    "Спорт": "sport",
    "Культура": "culture",
    "Интернет и СМИ": "media",
    "Ценности": "values",
    "Путешествия": "travel",
    "Из жизни": "life",
    "Среда обитания": "realty",
    "Забота о себе": "wellness",
    "Авто": "auto",
}

SLUG_TO_RUBRIC = {slug: rubric for rubric, slug in RUBRIC_TO_SLUG.items()}

# Какие рубрики качать
SELECTED_RUBRICS = [
    "Россия",
    "Мир",
    "Экономика",
    "Наука и техника",
    "Спорт",
    "Культура",
    # "Бывший СССР",
    # "Силовые структуры",
    # "Интернет и СМИ",
    # "Путешествия",
    # "Из жизни",
    # "Авто",
    # "Ценности",
    # "Среда обитания",
    # "Забота о себе",
]

# Если True, качать все рубрики из RUBRIC_TO_SLUG
USE_ALL_RUBRICS = False

# Если понадобится архив дня целиком
INCLUDE_GENERAL_ARCHIVE = False

# Защита от попадания на рейт-лимиты
CONCURRENCY = 8
PAUSE_SECONDS = 0.3

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)


@dataclass
class ArticleMeta:
    url: str
    source: str
    published_at: str
    category_raw: Optional[str]
    category_norm: Optional[str]
    archive_date: str


class LentaArchiveParser:
    """
    Парсер архива lenta.ru под текущую структуру:
    - общий архив дня: https://lenta.ru/YYYY/MM/DD/
    - архив рубрики за день: https://lenta.ru/rubrics/<slug>/YYYY/MM/DD/
    - пагинация: .../page/2/

    Ключевой принцип:
    - рубрику берем в первую очередь из положения новости в архиве:
      * если архив рубричный -> рубрика известна заранее;
      * если общий архив -> берем рубрику из хвоста карточки после времени.
    - на странице статьи рубрика только уточняется, но не переопределяется
      значением из глобального меню.
    """

    article_url_re = re.compile(
        r"^https?://lenta\.ru/news/\d{4}/\d{2}/\d{2}/[^/]+/?$"
    )
    time_re = re.compile(r"\b(\d{2}:\d{2})\b")

    def __init__(
        self,
        output_dir: str,
        output_filename: str,
        start_date: str,
        end_date: str,
        rubrics: list[str],
        concurrency: int = 8,
        pause_seconds: float = 0.3,
        include_general_archive: bool = False,
    ):
        self.output_dir = Path(output_dir)
        self.output_filename = output_filename
        self.output_path = self.output_dir / self.output_filename

        self.start_date = datetime.strptime(start_date, "%d.%m.%Y").date()
        self.end_date = datetime.strptime(end_date, "%d.%m.%Y").date()
        self.rubrics = rubrics
        self.concurrency = concurrency
        self.pause_seconds = pause_seconds
        self.include_general_archive = include_general_archive

        self._session: Optional[aiohttp.ClientSession] = None
        self._outfile = None
        self._csv_writer = None
        self._seen_urls: set[str] = set()
        self._saved_count = 0
        self._timeouts = aiohttp.ClientTimeout(total=60, connect=20, sock_read=60)
        self._sem = asyncio.Semaphore(concurrency)

    @property
    def session(self) -> aiohttp.ClientSession:
        if self._session is None or self._session.closed:
            connector = aiohttp.TCPConnector(limit=self.concurrency)
            self._session = aiohttp.ClientSession(
                connector=connector,
                timeout=self._timeouts,
                headers={"User-Agent": USER_AGENT},
            )
        return self._session

    @property
    def writer(self) -> csv.DictWriter:
        if self._csv_writer is None:
            self.output_dir.mkdir(parents=True, exist_ok=True)
            self._outfile = open(self.output_path, "w", newline="", encoding="utf-8")
            self._csv_writer = csv.DictWriter(
                self._outfile,
                fieldnames=[
                    "source",
                    "url",
                    "archive_date",
                    "published_at",
                    "title",
                    "text",
                    "category_raw",
                    "category_norm",
                    "collected_at",
                ],
            )
            self._csv_writer.writeheader()
        return self._csv_writer

    def iter_dates(self) -> Iterable[date]:
        current = self.start_date
        while current <= self.end_date:
            yield current
            current += timedelta(days=1)

    async def fetch_html(self, url: str) -> str:
        async with self._sem:
            for attempt in range(3):
                try:
                    async with self.session.get(url, allow_redirects=True) as response:
                        response.raise_for_status()
                        return await response.text()
                except (aiohttp.ClientError, asyncio.TimeoutError) as exc:
                    if attempt == 2:
                        raise
                    wait_time = 1.5 * (attempt + 1)
                    logger.warning("Retry %s after error on %s: %s", attempt + 1, url, exc)
                    await asyncio.sleep(wait_time)
            raise RuntimeError(f"Failed to fetch {url}")

    @staticmethod
    def build_archive_url(day: date, rubric_slug: Optional[str], page: int = 1) -> str:
        date_path = day.strftime("%Y/%m/%d")
        if rubric_slug:
            base = f"{BASE_URL}/rubrics/{rubric_slug}/{date_path}/"
        else:
            base = f"{BASE_URL}/{date_path}/"

        if page > 1:
            return urljoin(base, f"page/{page}/")
        return base

    @staticmethod
    def normalize_spaces(text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def get_requested_rubric_from_archive_url(archive_url: str) -> Optional[str]:
        """
        Если мы на странице /rubrics/<slug>/YYYY/MM/DD/, возвращаем рубрику по slug.
        """
        path_parts = [part for part in urlparse(archive_url).path.split("/") if part]
        # ожидаем ["rubrics", "<slug>", "2026", "04", "24", ...]
        if len(path_parts) >= 2 and path_parts[0] == "rubrics":
            return SLUG_TO_RUBRIC.get(path_parts[1])
        return None

    def parse_archive_page(
        self,
        html: str,
        archive_url: str,
        archive_day: date,
        requested_rubric_name: Optional[str],
    ) -> tuple[list[ArticleMeta], bool]:
        soup = BeautifulSoup(html, "html.parser")
        items: list[ArticleMeta] = []

        rubric_from_url = self.get_requested_rubric_from_archive_url(archive_url)
        fixed_archive_rubric = requested_rubric_name or rubric_from_url

        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            abs_url = urljoin(BASE_URL, href)

            if not self.article_url_re.match(abs_url):
                continue

            if abs_url in self._seen_urls:
                continue

            text = self.normalize_spaces(a.get_text(" ", strip=True))
            if not text:
                continue

            time_match = self.time_re.search(text)
            published_at = None
            if time_match:
                hhmm = time_match.group(1)
                published_at = f"{archive_day.isoformat()} {hhmm}:00"

            # На рубричном архиве рубрика фиксирована самой страницей.
            # На общем архиве пробуем достать ее из хвоста карточки после времени.
            if fixed_archive_rubric:
                category_raw = fixed_archive_rubric
            else:
                category_raw = self.extract_category_from_archive_text(text)

            items.append(
                ArticleMeta(
                    url=abs_url,
                    source="lenta.ru",
                    published_at=published_at or f"{archive_day.isoformat()} 00:00:00",
                    category_raw=category_raw,
                    category_norm=category_raw,
                    archive_date=archive_day.isoformat(),
                )
            )
            self._seen_urls.add(abs_url)

        has_next = False
        for a in soup.find_all("a", href=True):
            label = self.normalize_spaces(a.get_text(" ", strip=True)).lower()
            href = a["href"]
            if "дальше" in label and "/page/" in href:
                has_next = True
                break

        return items, has_next

    def extract_category_from_archive_text(self, text: str) -> Optional[str]:
        """
        Для общего архива формат обычно такой:
        "<заголовок> 00:31 Мир"
        Берем точное совпадение хвоста после времени с известной рубрикой.
        """
        time_match = self.time_re.search(text)
        if not time_match:
            return None

        tail = text[time_match.end():].strip(" -–—")
        if not tail:
            return None

        for rubric in RUBRIC_TO_SLUG.keys():
            if tail == rubric:
                return rubric

        return None

    def parse_article_page(self, html: str, fallback_meta: ArticleMeta) -> dict:
        soup = BeautifulSoup(html, "html.parser")

        title = self.extract_title(soup)
        text = self.extract_text(soup)
        published_at = self.extract_published_at(soup) or fallback_meta.published_at

        # Рубрику из статьи берем только из локального верхнего блока статьи.
        # Если не нашли — оставляем то, что уже известно из архива.
        article_category = self.extract_article_category(soup)
        category_raw = article_category or fallback_meta.category_raw

        return {
            "source": fallback_meta.source,
            "url": fallback_meta.url,
            "archive_date": fallback_meta.archive_date,
            "published_at": published_at,
            "title": title,
            "text": text,
            "category_raw": category_raw,
            "category_norm": category_raw,
            "collected_at": datetime.now(UTC).replace(microsecond=0).isoformat(),
        }

    @staticmethod
    def extract_title(soup: BeautifulSoup) -> Optional[str]:
        h1 = soup.find("h1")
        if h1:
            return LentaArchiveParser.normalize_spaces(h1.get_text(" ", strip=True))

        meta_og = soup.find("meta", attrs={"property": "og:title"})
        if meta_og and meta_og.get("content"):
            return LentaArchiveParser.normalize_spaces(meta_og["content"])

        if soup.title:
            return LentaArchiveParser.normalize_spaces(soup.title.get_text(" ", strip=True))
        return None

    @staticmethod
    def extract_text(soup: BeautifulSoup) -> Optional[str]:
        candidate_blocks: list[Tag] = []

        selectors = [
            {"attrs": {"itemprop": "articleBody"}},
            {"name": "div", "attrs": {"class": re.compile(r"topic-body|article-body|js-topic__text")}},
            {"name": "section", "attrs": {"class": re.compile(r"topic-body|article-body")}},
            {"name": "article"},
            {"name": "main"},
        ]

        for selector in selectors:
            node = soup.find(selector.get("name"), attrs=selector.get("attrs"))
            if node:
                candidate_blocks.append(node)

        for block in candidate_blocks:
            paragraphs = [
                LentaArchiveParser.normalize_spaces(p.get_text(" ", strip=True))
                for p in block.find_all(["p", "div"])
            ]
            paragraphs = [p for p in paragraphs if len(p) > 40]
            if paragraphs:
                return "\n".join(paragraphs)

        paragraphs = [
            LentaArchiveParser.normalize_spaces(p.get_text(" ", strip=True))
            for p in soup.find_all("p")
        ]
        paragraphs = [p for p in paragraphs if len(p) > 40]
        if paragraphs:
            return "\n".join(paragraphs)

        return None

    @staticmethod
    def extract_published_at(soup: BeautifulSoup) -> Optional[str]:
        time_tag = soup.find("time")
        if time_tag and time_tag.get("datetime"):
            return time_tag["datetime"]

        for attr_name, attr_value in [
            ("property", "article:published_time"),
            ("property", "og:published_time"),
            ("name", "publish_date"),
            ("name", "pubdate"),
        ]:
            tag = soup.find("meta", attrs={attr_name: attr_value})
            if tag and tag.get("content"):
                return tag["content"]

        full_text = soup.get_text(" ", strip=True)
        m = re.search(
            r"\b(\d{2}:\d{2}),\s+(\d{1,2}\s+[А-Яа-яё]+\s+\d{4})\b",
            full_text,
            flags=re.IGNORECASE,
        )
        if m:
            hhmm, rus_date = m.groups()
            try:
                dt = parse_russian_datetime(f"{rus_date} {hhmm}")
                return dt.isoformat(sep=" ")
            except ValueError:
                return None

        return None

    @staticmethod
    def extract_article_category(soup: BeautifulSoup) -> Optional[str]:
        """
        Ищем рубрику только в верхнем блоке статьи, а не по всей странице:
        на странице статьи рубрика находится рядом со временем публикации.
        Например:
        00:31, 24 апреля 2026 | Мир
        """
        # 1) Найти time-блок и посмотреть ближайшие ссылки/контейнеры вокруг него
        time_tag = soup.find("time")
        if time_tag:
            parent = time_tag.parent
            for _ in range(4):
                if parent is None:
                    break

                # сначала ищем прямые ссылки в локальном контейнере
                local_links = parent.find_all("a", href=True)
                for a in local_links:
                    text = LentaArchiveParser.normalize_spaces(a.get_text(" ", strip=True))
                    if text in RUBRIC_TO_SLUG:
                        return text

                parent = parent.parent

        # 2) Ищем в верхней части страницы до заголовка h1, чтобы не захватить меню и футер
        h1 = soup.find("h1")
        if h1:
            prefix_text_parts = []
            for el in soup.find_all():
                if el == h1:
                    break
                if el.name in {"script", "style"}:
                    continue
                text = LentaArchiveParser.normalize_spaces(el.get_text(" ", strip=True))
                if text:
                    prefix_text_parts.append(text)

            prefix_text = " | ".join(prefix_text_parts[-20:])  # только ближайший верхний кусок
            for rubric in RUBRIC_TO_SLUG:
                if re.search(rf"\b{re.escape(rubric)}\b", prefix_text):
                    return rubric

        # 3) Явный fallback: иногда рубрика видна как отдельный короткий заголовок над h1
        body_text = soup.get_text("\n", strip=True)
        lines = [LentaArchiveParser.normalize_spaces(line) for line in body_text.splitlines()]
        lines = [line for line in lines if line]

        try:
            h1_text = LentaArchiveParser.normalize_spaces(h1.get_text(" ", strip=True)) if h1 else None
            if h1_text and h1_text in lines:
                idx = lines.index(h1_text)
                window = lines[max(0, idx - 6):idx]
                for line in reversed(window):
                    if line in RUBRIC_TO_SLUG:
                        return line
        except Exception:
            pass

        return None

    async def fetch_and_parse_article(self, meta: ArticleMeta) -> Optional[dict]:
        try:
            html = await self.fetch_html(meta.url)
        except Exception as exc:
            logger.warning("Cannot fetch article %s: %s", meta.url, exc)
            return None

        try:
            row = self.parse_article_page(html, meta)
        except Exception as exc:
            logger.warning("Cannot parse article %s: %s", meta.url, exc)
            return None

        if not row.get("title") or not row.get("text"):
            logger.warning("Article parsed poorly, skip %s", meta.url)
            return None

        return row

    async def process_archive_day(self, day: date, rubric_name: Optional[str]) -> int:
        rubric_slug = RUBRIC_TO_SLUG[rubric_name] if rubric_name else None

        page = 1
        total_saved_for_day = 0

        while True:
            archive_url = self.build_archive_url(day, rubric_slug, page)

            try:
                html = await self.fetch_html(archive_url)
            except Exception as exc:
                logger.warning("Cannot fetch archive page %s: %s", archive_url, exc)
                break

            metas, has_next = self.parse_archive_page(
                html=html,
                archive_url=archive_url,
                archive_day=day,
                requested_rubric_name=rubric_name,
            )

            if not metas:
                logger.info("No news at %s", archive_url)
                break

            tasks = [asyncio.create_task(self.fetch_and_parse_article(meta)) for meta in metas]
            rows = await asyncio.gather(*tasks)

            rows_to_write = [row for row in rows if row is not None]
            if rows_to_write:
                self.writer.writerows(rows_to_write)
                total_saved_for_day += len(rows_to_write)
                self._saved_count += len(rows_to_write)

            logger.info(
                "Processed %s | rubric=%s | page=%s | saved=%s | total=%s",
                day.isoformat(),
                rubric_name or "ALL",
                page,
                len(rows_to_write),
                self._saved_count,
            )

            if not has_next:
                break

            page += 1
            await asyncio.sleep(self.pause_seconds)

        return total_saved_for_day

    async def run(self) -> Path:
        try:
            for day in self.iter_dates():
                if self.include_general_archive:
                    await self.process_archive_day(day, rubric_name=None)
                    await asyncio.sleep(self.pause_seconds)

                for rubric_name in self.rubrics:
                    await self.process_archive_day(day, rubric_name=rubric_name)
                    await asyncio.sleep(self.pause_seconds)
        finally:
            await self.shutdown()

        return self.output_path

    async def shutdown(self) -> None:
        if self._session is not None and not self._session.closed:
            await self._session.close()

        if self._outfile is not None:
            self._outfile.close()

        logger.info("Done. Saved %s rows to %s", self._saved_count, self.output_path)


def parse_russian_datetime(value: str) -> datetime:
    months = {
        "января": 1,
        "февраля": 2,
        "марта": 3,
        "апреля": 4,
        "мая": 5,
        "июня": 6,
        "июля": 7,
        "августа": 8,
        "сентября": 9,
        "октября": 10,
        "ноября": 11,
        "декабря": 12,
    }

    m = re.match(r"(\d{1,2})\s+([А-Яа-яё]+)\s+(\d{4})\s+(\d{2}:\d{2})", value)
    if not m:
        raise ValueError(f"Cannot parse russian datetime: {value}")

    day_str, month_str, year_str, hhmm = m.groups()
    month = months[month_str.lower()]
    return datetime.strptime(
        f"{year_str}-{month:02d}-{int(day_str):02d} {hhmm}",
        "%Y-%m-%d %H:%M",
    )


def validate_rubrics(selected_rubrics: list[str]) -> list[str]:
    unknown = [rubric for rubric in selected_rubrics if rubric not in RUBRIC_TO_SLUG]
    if unknown:
        raise ValueError(
            f"Unknown rubrics: {unknown}. "
            f"Available: {list(RUBRIC_TO_SLUG.keys())}"
        )
    return selected_rubrics


async def run_lenta_parser(
    output_dir: str,
    output_filename: str,
    start_date: str,
    end_date: str,
    selected_rubrics: Optional[list[str]] = None,
    use_all_rubrics: bool = False,
    include_general_archive: bool = False,
    concurrency: int = 8,
    pause_seconds: float = 0.3,
) -> Path:
    rubrics = list(RUBRIC_TO_SLUG.keys()) if use_all_rubrics else (selected_rubrics or [])
    rubrics = validate_rubrics(rubrics)

    parser = LentaArchiveParser(
        output_dir=output_dir,
        output_filename=output_filename,
        start_date=start_date,
        end_date=end_date,
        rubrics=rubrics,
        concurrency=concurrency,
        pause_seconds=pause_seconds,
        include_general_archive=include_general_archive,
    )

    logger.info(
        "Start parse: %s -> %s | rubrics=%s | output=%s",
        start_date,
        end_date,
        rubrics,
        parser.output_path,
    )
    return await parser.run()

In [ ]:
saved_path = await run_lenta_parser(
    output_dir="../news_data",
    output_filename="01012025-30092025_newsdata.csv",
    start_date="01.01.2025",
    end_date="30.09.2025",
    selected_rubrics=SELECTED_RUBRICS,
    use_all_rubrics=USE_ALL_RUBRICS,
    include_general_archive=INCLUDE_GENERAL_ARCHIVE,
    concurrency=CONCURRENCY,
    pause_seconds=PAUSE_SECONDS,
)

saved_path

In [32]:
# Преобразование файлов .parquet

from pathlib import Path
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    raise ImportError("Нужен pyarrow. Установи: %pip install pyarrow")

# =========================
# НАСТРОЙКИ
# =========================

INPUT_FILES = [
    r"..\news_data\01012025-30092025_newsdata.csv",
    r"..\news_data\01102025-23042026_newsdata.csv",
]

OUTPUT_PARQUET = r"..\news_data\0101202-23042026_merged_newsdata.parquet"

# Если разделитель не запятая, поменяй здесь, например sep=';'
CSV_SEP = ","

# Размер чанка: можно увеличить/уменьшить в зависимости от RAM
CHUNK_SIZE = 100_000

# Сжатие parquet
COMPRESSION = "snappy"


# =========================
# ОБЪЕДИНЕНИЕ CSV -> PARQUET
# =========================

input_paths = [Path(p) for p in INPUT_FILES]
output_path = Path(OUTPUT_PARQUET)
output_path.parent.mkdir(parents=True, exist_ok=True)

for p in input_paths:
    if not p.exists():
        raise FileNotFoundError(f"Файл не найден: {p}")

writer = None
all_columns = None
total_rows = 0

try:
    for file_idx, csv_path in enumerate(input_paths, start=1):
        print(f"\n[{file_idx}/{len(input_paths)}] Читаю: {csv_path}")

        chunk_iter = pd.read_csv(
            csv_path,
            sep=CSV_SEP,
            chunksize=CHUNK_SIZE,
            low_memory=False,
        )

        for chunk_idx, chunk in enumerate(chunk_iter, start=1):
            # На первом чанке фиксируем набор колонок
            if all_columns is None:
                all_columns = list(chunk.columns)
            else:
                for col in all_columns:
                    if col not in chunk.columns:
                        chunk[col] = pd.NA

                extra_cols = [c for c in chunk.columns if c not in all_columns]
                if extra_cols:
                    # добавляем новые колонки в общий список
                    all_columns.extend(extra_cols)

                for col in all_columns:
                    if col not in chunk.columns:
                        chunk[col] = pd.NA

                chunk = chunk[all_columns]

            chunk = chunk[all_columns]

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(
                    output_path,
                    table.schema,
                    compression=COMPRESSION,
                )

            writer.write_table(table)
            total_rows += len(chunk)

            print(
                f"  chunk {chunk_idx}: +{len(chunk):,} строк "
                f"(всего {total_rows:,})"
            )

finally:
    if writer is not None:
        writer.close()

print("\nГотово.")
print(f"Итоговый parquet: {output_path.resolve()}")
print(f"Всего строк: {total_rows:,}")


[1/2] Читаю: ..\news_data\01012025-30092025_newsdata.csv
  chunk 1: +85,459 строк (всего 85,459)

[2/2] Читаю: ..\news_data\01102025-23042026_newsdata.csv
  chunk 1: +61,161 строк (всего 146,620)

Готово.
Итоговый parquet: C:\Users\skuts\vibecode_projects\news_data\0101202-23042026_merged_newsdata.parquet
Всего строк: 146,620
